In [1]:
import torch
import torch.nn as nn
torch.manual_seed(42)

# (예시) 4토큰짜리 문장이라고 가정
# 실제로는 토크나이저가 IDs를 만듭니다. 여기서는 임의 IDs로 시연
token_ids = torch.tensor([[3, 7, 2, 9]])  # [batch=1, seq_len=4]

vocab_size = 100
d_model = 16   # 임베딩 차원 (작게 해서 보기 쉽게)
n_heads  = 2
head_dim = d_model // n_heads  # 8

embed = nn.Embedding(vocab_size, d_model)

# 입력 임베딩 (X): [B, T, d_model]
X = embed(token_ids)  # shape: [1, 4, 16]
print("X shape:", X.shape)


X shape: torch.Size([1, 4, 16])


In [2]:
# Q, K, V를 만드는 가중치 (학습 대상 파라미터)
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

# Q,K,V 생성
Q = W_Q(X)  # [1, 4, 16]
K = W_K(X)  # [1, 4, 16]
V = W_V(X)  # [1, 4, 16]

print("Q/K/V shape:", Q.shape, K.shape, V.shape)


Q/K/V shape: torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])


In [3]:
# [B, T, d_model] -> [B, n_heads, T, head_dim]
def split_heads(t):
    B, T, D = t.shape
    return t.view(B, T, n_heads, head_dim).transpose(1, 2)  # [B, H, T, Dh]

Qh = split_heads(Q)
Kh = split_heads(K)
Vh = split_heads(V)

print("Qh/Kh/Vh shape:", Qh.shape, Kh.shape, Vh.shape)  # [1, 2, 4, 8]


Qh/Kh/Vh shape: torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8]) torch.Size([1, 2, 4, 8])


In [4]:
# 어텐션 스코어: Q * K^T / sqrt(Dh)
Dh = head_dim
scores = torch.matmul(Qh, Kh.transpose(-2, -1)) / (Dh ** 0.5)  # [B, H, T, T]

# 소프트맥스로 가중치
attn = scores.softmax(dim=-1)  # [B, H, T, T]

# 가중합으로 컨텍스트 계산
context = torch.matmul(attn, Vh)  # [B, H, T, Dh]

print("scores shape:", scores.shape)
print("attn shape  :", attn.shape)
print("context shape:", context.shape)


scores shape: torch.Size([1, 2, 4, 4])
attn shape  : torch.Size([1, 2, 4, 4])
context shape: torch.Size([1, 2, 4, 8])


In [5]:
# [B, H, T, Dh] -> [B, T, H*Dh] = [B, T, d_model]
context_cat = context.transpose(1, 2).contiguous().view(X.size(0), X.size(1), d_model)

# 멀티헤드 출력에 적용하는 최종 선형 (통상 Wo)
Wo = nn.Linear(d_model, d_model, bias=False)
out = Wo(context_cat)  # [B, T, d_model]

print("context_cat shape:", context_cat.shape)
print("out shape        :", out.shape)


context_cat shape: torch.Size([1, 4, 16])
out shape        : torch.Size([1, 4, 16])


In [6]:
# 첫 번째 헤드의 어텐션 가중치 행렬을 확인해보자
# attn[batch=0, head=0] -> [T, T]
with torch.no_grad():
    print("=== Head 0 Attention Weights (T x T) ===")
    print(attn[0, 0])

# 첫 토큰의 Q/K/V 일부 값
with torch.no_grad():
    print("\nQ[0,0,:5] =", Q[0,0,:5])
    print("K[0,0,:5] =", K[0,0,:5])
    print("V[0,0,:5] =", V[0,0,:5])


=== Head 0 Attention Weights (T x T) ===
tensor([[0.1781, 0.1899, 0.3267, 0.3054],
        [0.1991, 0.1967, 0.4053, 0.1989],
        [0.1960, 0.2229, 0.3022, 0.2789],
        [0.1994, 0.2042, 0.3699, 0.2265]], requires_grad=True)

Q[0,0,:5] = tensor([-0.4948, -0.7565, -0.8726, -0.5964, -0.2065], requires_grad=True)
K[0,0,:5] = tensor([ 0.7441,  0.1410, -0.2345, -0.7178,  0.1494], requires_grad=True)
V[0,0,:5] = tensor([-0.5110, -0.4519,  0.2230, -0.4761, -0.1019], requires_grad=True)


In [15]:
# !pip install transformers kobert-transformers

In [18]:
from kobert_transformers import get_tokenizer
from transformers import BertModel

tokenizer = get_tokenizer()
model = BertModel.from_pretrained('skt/kobert-base-v1')

tokens = tokenizer("오늘 날씨가 참 좋네요", return_tensors='pt')
outputs = model(**tokens)

print(tokens)


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ekfla\.cache\huggingface\hub\models--monologg--kobert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_down

{'input_ids': tensor([[   2, 3419, 1408, 5330, 4427, 4204, 5703,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [53]:
import re
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

# ============================================================
# 1. 텍스트 정규화 함수
# ============================================================
def normalize_korean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"<[^>]+>", " ", text)               # HTML 태그 제거
    text = re.sub(r"http\S+|www\S+", " ", text)        # URL 제거
    text = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ .,!?\"'’‘…~\-]", " ", text)  # 특수문자 제거
    text = re.sub(r"([ㅋㅎㅠ])\1{1,}", r"\1\1", text)  # ㅋㅋㅋ, ㅎㅎㅎ → ㅋㅋ, ㅎㅎ
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

# ============================================================
# 2. 데이터 로드 및 전처리
# ============================================================
data_path = "data/ratings_train.txt"  # 경로 확인 필요

df = pd.read_csv(data_path, sep="\t").head(1000)
df = df.dropna(subset=["document"])
df["document"] = df["document"].apply(normalize_korean_text)
df = df[df["document"].str.len() > 1]  # 1자 이하 제거
df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
df = df[df["label"].isin([0, 1])]
df["label"] = df["label"].astype("int64")

print("샘플 확인:")
print(df.sample(5))

# train/test 분리
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# ============================================================
# 3. KoBERT 모델 + Tokenizer
# ============================================================
MODEL_NAME = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tok_fn(batch):
    return tokenizer(batch["document"], truncation=True, max_length=128)

train_tok = train_ds.map(tok_fn, batched=True, remove_columns=["id", "document"])
test_tok  = test_ds.map(tok_fn,  batched=True, remove_columns=["id", "document"])

# ============================================================
# 4. BertModel + Linear Head 정의
# ============================================================
class BertClsHead(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.backbone = BertModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]  # [CLS] 토큰
        logits = self.classifier(self.dropout(pooled))
        result = {"logits": logits}
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result["loss"] = loss
        return result

model = BertClsHead(MODEL_NAME, num_labels=2)

# ============================================================
# 5. 평가 함수
# ============================================================
def metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pred), "f1": f1_score(y, pred)}

# ============================================================
# 6. 학습 설정
# ============================================================
args = TrainingArguments(
    output_dir="./kobert_from_bertmodel",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    use_mps_device=(torch.backends.mps.is_available() if not torch.cuda.is_available() else False),
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    compute_metrics=metrics,
)

trainer.train()

# ============================================================
# 7. 평가 및 예측 테스트
# ============================================================
eval_res = trainer.evaluate()
print("평가 결과:", eval_res)

samples = ["정말 감동적인 영화였습니다.", "지루하고 시간 낭비였어요."]
enc = tokenizer(samples, return_tensors="pt", padding=True, truncation=True).to(model.classifier.weight.device)
with torch.no_grad():
    out = model(**enc)
    probs = torch.softmax(out["logits"], dim=-1).cpu().numpy()

for s, p in zip(samples, probs):
    print(f"[{s}] → 부정={p[0]:.3f}, 긍정={p[1]:.3f}, 예측={p.argmax()}")


샘플 확인:
           id                                           document  label
836   9121447                               꿀잼! 특히 소유랑 서인영ㅋㅋ매력터져      1
974   8825761                                    tv 전기세가 아까웠다!!!      0
96    9361974                                         재미있는영화입니다.      1
590   9589263                                 90년도로 돌아가게 해준다는 ㅡㅡ      0
452  10131631  이 영화의 3박자는 디테일이었다. 정말 시가전의 퀄리티는 완벽했고 남자의 사랑에 대...      1


Map: 100%|██████████| 200/200 [00:00<00:00, 20253.04 examples/s]
C:\Users\ekfla\AppData\Local\Temp\ipykernel_11948\2301277758.py:112: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.701400,0.681603,0.540000,0.163636
2,0.567600,0.577185,0.700000,0.700000


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


평가 결과: {'eval_loss': 0.5771846771240234, 'eval_accuracy': 0.7, 'eval_f1': 0.7, 'eval_runtime': 4.0291, 'eval_samples_per_second': 49.639, 'eval_steps_per_second': 3.227, 'epoch': 2.0}
[정말 감동적인 영화였습니다.] → 부정=0.088, 긍정=0.912, 예측=1
[지루하고 시간 낭비였어요.] → 부정=0.588, 긍정=0.412, 예측=0
